# ドラマ vectorization and recommender

Import packages

In [1]:
import pandas as pd
import ast

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Import Data

In [2]:
df_raw = pd.read_pickle('df_wiki.pkl')

Data Preprocessing Area

In [3]:
df = df_raw.copy()

df['hour'] = df.time.apply(lambda x: x[0]).str.extract(r'^([0-9]+)時')
df['time_start'] = pd.to_datetime(df.time.apply(lambda x: x[0]), format= '%H時%M分', errors='coerce')
df['time_end'] = pd.to_datetime(df.time.apply(lambda x: x[1] if len(x) > 1 else None), format= '|%H時%M分', errors='coerce')
df['min_length'] = (df['time_end'] - df['time_start']).astype('timedelta64[m]')

# Make up after calculation
df['time_start'] = df['time_start'].dt.time
df['time_end'] = df['time_end'].dt.time

df = df.drop(['date', 'time', 'url_wiki_zh', 'url', 'theme_song'], axis=1)

In [4]:
df.query('name_ja == "失恋ショコラティエ"')

,weekday,name_zh,name_ja,url_wiki_ja,TV,char_main,char_others,year,season,ジャンル,原作,脚本,監督,出演者,オープニング,エンディング,プロデューサー,制作,放送チャンネル,音声形式,放送国・地域,放送時間,放送期間,放送分,回数,演出,放送枠,ナレーター,製作総指揮,映像形式,企画,原案,シーズン数,製作,音楽,時代設定,チーフ・,作,制作統括,声の出演,言語,放送局,プロデュース,国・地域,話数,撮影地,撮影監督,編集,別名,シリーズ数,監修,配信サイト,配信期間,続編,配信国・地域,配信時間,各話の長さ,前作,制作プロデューサー,関連番組,エグゼクティブ・,テーマ曲作者,作曲,総監督,ディレクター,構成,hour,time_start,time_end,min_length
1,一,失戀巧克力職人,失恋ショコラティエ,https://ja.wikipedia.org/wiki/%E5%A4%B1%E6%81%8B%E3%82%B7%E3%83%A7%E3%82%B3%E3%83%A9%E3%83%86%E3%82%A3%E3%82%A8,[CX],[松本潤],"[石原聰美, 溝端淳平, 水川麻美, 佐藤隆太, 有村架純, 加藤成亮, 水原希子, 眞島秀和, 竹中直人]",2014,1,[テレビドラマ],[水城せとな],"[安達奈緒子, 越川美埜子]",NaN,"[松本潤, 石原さとみ, 水川あさみ, 水原希子, 溝端淳平, 有村架純, 佐藤隆太, 竹中直人, 加藤シゲアキ, 眞島秀和, 夏月, 佐藤祐基, 若月佑美, 千葉雄大]",[Ken Arai「Marble」],[嵐「Bittersweet」],"[若松央樹, 小原一隆]",[フジテレビ],NaN,NaN,[日本],[月曜日21:00 - 21:54],[2014年1月13日-3月24日],[54分],[11],"[松山博昭, 宮木正悟, 関野宗紀, 品田俊介, 小原一隆]",[フジテレビ月曜9時枠の連続ドラマ],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21,21:00:00,21:54:00,54.0


In [5]:
df.loc[df.ジャンル.isnull()][['name_ja', 'year', 'season', 'hour', 'min_length', 'TV', '出演者', 'url_wiki_ja']]

,name_ja,year,season,hour,min_length,TV,出演者,url_wiki_ja
5,紙の月,2014,1,22,48.0,[NHK],NaN,https://ja.wikipedia.org/wiki/%E3%83%86%E3%83%AC%E3%83%93%E3%83%89%E3%83%A9%E3%83%9E
8,今夜は心だけ抱いて,2014,1,23,29.0,"[田中美佐子, 土屋太鳳]",NaN,https://ja.wikipedia.org/wiki/%E4%BB%8A%E5%A4%9C%E3%81%AF%E5%BF%83%E3%81%A0%E3%81%91%E6%8A%B1%E3%81%84%E3%81%A6
14,テレビ東京開局50周年特別企画新春ワイド時代劇 影武者徳川家康,2014,1,18,300.0,[TX],NaN,https://ja.wikipedia.org/wiki/%E6%96%B0%E6%98%A5%E3%83%AF%E3%82%A4%E3%83%89%E6%99%82%E4%BB%A3%E5%8A%87
15,科捜研の女Season13,2014,1,19,56.0,[EX],NaN,https://ja.wikipedia.org/wiki/%E6%A8%8B%E5%8F%A3%E9%9A%86%E5%89%87
20,慰謝料弁護士〜あなたの涙、お金に変えましょう〜,2014,1,23,NaN,[YTV],"[田中直樹, 矢田亜希子, 渡辺直美, 山田菜々, 岩﨑名美, 高嶋香帆, 美保純]",https://ja.wikipedia.org/wiki/%E6%85%B0%E8%AC%9D%E6%96%99%E5%BC%81%E8%AD%B7%E5%A3%AB%E3%80%9C%E3%81%82%E3%81%AA%E3%81%9F%E3%81%AE%E6%B6%99%E3%80%81%E3%81%8A%E9%87%91%E3%81%AB%E5%A4%89%E3%81%88%E3%...
...,...,...,...,...,...,...,...,...
1165,きれいのくに,2021,2,20,43.0,[NHK],"[吉田羊, 蓮佛美沙子, 平原テツ, 小野花梨, 橋本淳, 加藤ローサ, 青木柚, 見上愛, 岡本夏美, 山脇辰哉, 秋元龍太朗, 稲垣吾郎]",https://ja.wikipedia.org/wiki/%E3%81%8D%E3%82%8C%E3%81%84%E3%81%AE%E3%81%8F%E3%81%AB
1170,著飾る戀には理由があって,2021,2,22,57.0,[TBS],NaN,https://ja.wikipedia.org/wiki/%E7%99%BE%E4%BA%BA%E4%B8%80%E9%A6%96
1184,70才、初めて産みますセブンティウイザン。,2021,2,22,43.0,[NHK],"[小日向文世, 竹下景子, 伊藤歩, 矢本悠馬, 臼田あさ美, 蔵下穂波, 田中美央, 笑福亭鶴光, 山本圭祐, 竜雷太, ふせえり, 前田亜季, 吉澤健, 根岸季衣, 宇崎竜童, 中村梅雀]",https://ja.wikipedia.org/wiki/%E3%82%BB%E3%83%96%E3%83%B3%E3%83%86%E3%82%A3%E3%82%A6%E3%82%A4%E3%82%B6%E3%83%B3
1193,泣くな研修醫,2021,2,23,30.0,[EX],NaN,


In [6]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1206 entries, 0 to 1205
Data columns (total 70 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   weekday      1206 non-null   object 
 1   name_zh      1206 non-null   object 
 2   name_ja      1206 non-null   object 
 3   url_wiki_ja  1206 non-null   object 
 4   TV           1206 non-null   object 
 5   char_main    1206 non-null   object 
 6   char_others  1206 non-null   object 
 7   year         1206 non-null   int64  
 8   season       1206 non-null   int64  
 9   ジャンル         1017 non-null   object 
 10  原作           545 non-null    object 
 11  脚本           961 non-null    object 
 12  監督           364 non-null    object 
 13  出演者          1025 non-null   object 
 14  オープニング       305 non-null    object 
 15  エンディング       600 non-null    object 
 16  プロデューサー      819 non-null    object 
 17  制作           841 non-null    object 
 18  放送チャンネル      290 non-null    object 
 19  音声形式  

In [7]:
for col in df.columns:
    print(f"{col} value counts = {df[col].value_counts()}")

weekday value counts = 六       241
日       227
四       171
五       165
三       129
二       125
一        58
一至五      27
一至六      25
六至日      13
          4
三至四       3
一至四       2
日至一       2
三至日       2
五至六       2
二至六       2
一至二       1
二至四       1
日四        1
二至五       1
一至日       1
四至五       1
一至三       1
六三至四      1
Name: weekday, dtype: int64
name_zh value counts = 烈車戰隊特急者                              5
麒麟來了                                 4
直衝青天                                 4
真田丸                                  4
花燃燒                                  4
                                    ..
女科调员Season13                         1
是生是死還是父親（日语：生きるとか死ぬとか父親とか#テレビドラマ）    1
打耳光！～律師事務員箕輪用愛解決一切～                  1
半澤直樹2                                1
時間巡邏的OL                              1
Name: name_zh, Length: 1125, dtype: int64
name_ja value counts = 烈車戦隊トッキュウジャー           5
いだてん〜東京オリムピック噺〜        4
科捜研の女 第19シリーズ          4
真田丸                    4
青天を衝け                  4
       